<a href="https://colab.research.google.com/github/Rexi15/jupyter/blob/master/yolo_python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install ultralytics

In [3]:
"""
Step 1: baseline mAP for a tiny YOLO, FP32 vs fake-quantized fixed point.

Setup:
    pip install ultralytics torch
Run:
    python baseline_yolo_q115.py --model yolov3-tinyu.pt --data kitti.yaml
    python baseline_yolo_q115.py --model yolov3-tinyu.pt --data kitti.yaml --frac-bits 15
    python baseline_yolo_q115.py --model yolov3-tinyu.pt --data kitti.yaml --frac-bits 7

Notes:
  * kitti.yaml is the Ultralytics KITTI detection set (auto-downloads).
    It is single images, fine for baseline mAP. For the temporal-reuse
    study (steps 2-4) you will need KITTI raw/tracking or BDD100K video.
  * Q1.15 covers [-1, 1). Weights/activations outside that range clip.
    The range report below tells you how much clipping to expect and
    how many integer bits each layer really needs.
"""
import argparse
import json

import torch
import torch.nn as nn
from ultralytics import YOLO


def quantize_tensor(t: torch.Tensor, frac_bits: int, int_bits: int = 1):
    """Fake-quantize to signed Q(int_bits).(frac_bits) with saturation."""
    scale = 2.0 ** frac_bits
    hi = 2.0 ** (int_bits - 1) - 1.0 / scale
    lo = -(2.0 ** (int_bits - 1))
    return torch.clamp(torch.round(t * scale) / scale, lo, hi)


def weight_range_report(net: nn.Module):
    rows = []
    for name, m in net.named_modules():
        if isinstance(m, nn.Conv2d):
            w = m.weight.detach()
            rows.append({
                "layer": name,
                "max_abs_w": float(w.abs().max()),
                "frac_outside_q1": float((w.abs() >= 1.0).float().mean()),
            })
    return rows


def fake_quant_weights(net: nn.Module, frac_bits: int, int_bits: int):
    with torch.no_grad():
        for m in net.modules():
            if isinstance(m, nn.Conv2d):
                m.weight.copy_(quantize_tensor(m.weight, frac_bits, int_bits))
                if m.bias is not None:
                    m.bias.copy_(quantize_tensor(m.bias, frac_bits, int_bits))


def add_activation_quant(net: nn.Module, frac_bits: int, int_bits: int):
    handles = []
    for m in net.modules():
        if isinstance(m, nn.Conv2d):
            handles.append(m.register_forward_hook(
                lambda mod, inp, out: quantize_tensor(out, frac_bits, int_bits)))
    return handles


def evaluate(model: YOLO, data: str, imgsz: int, batch: int):
    r = model.val(data=data, imgsz=imgsz, batch=batch, verbose=False)
    return {"mAP50": float(r.box.map50), "mAP50-95": float(r.box.map)}


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", default="yolov3-tinyu.pt")
    ap.add_argument("--data", default="kitti.yaml")
    ap.add_argument("--imgsz", type=int, default=640)
    ap.add_argument("--batch", type=int, default=16)
    ap.add_argument("--frac-bits", type=int, default=None,
                    help="fractional bits for fake-quant, e.g. 15 or 7")
    ap.add_argument("--int-bits", type=int, default=1,
                    help="integer bits incl. sign (1 = Q1.x, range [-1,1))")
    ap.add_argument("--out", default="baseline_results.json")
    args = ap.parse_args(ARGS)

    results = {}

    # 1) FP32 baseline
    model = YOLO(args.model)
    results["fp32"] = evaluate(model, args.data, args.imgsz, args.batch)
    print("FP32:", results["fp32"])

    # 2) Optional fixed-point emulation
    if args.frac_bits is not None:
        model = YOLO(args.model)
        model.fuse()  # fold BN into conv so weights match what hardware sees
        net = model.model
        results["weight_ranges"] = weight_range_report(net)
        worst = sorted(results["weight_ranges"],
                       key=lambda r: -r["max_abs_w"])[:5]
        print("Layers with the largest |w| (need more integer bits):")
        for r in worst:
            print(f"  {r['layer']}: max|w|={r['max_abs_w']:.3f}, "
                  f"outside Q1 range={r['frac_outside_q1']:.2%}")

        fake_quant_weights(net, args.frac_bits, args.int_bits)
        add_activation_quant(net, args.frac_bits, args.int_bits)
        key = f"Q{args.int_bits}.{args.frac_bits}"
        results[key] = evaluate(model, args.data, args.imgsz, args.batch)
        print(f"{key}:", results[key])

    with open(args.out, "w") as f:
        json.dump(results, f, indent=2)
    print("Saved", args.out)




Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings


In [4]:
ARGS = ["--model", "yolov3-tinyu.pt", "--data", "kitti.yaml", "--out", "fp32.json"]
main()

Ultralytics 8.4.211 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLOv3-tiny summary (fused): 34 layers, 12,168,784 parameters, 0 gradients, 19.0 GFLOPs

WARNING ⚠️ Dataset 'kitti.yaml' images not found, missing path '/content/datasets/kitti/images/val'
Unzipping /content/datasets/kitti.zip to /content/datasets/kitti...: 100% ━━━━━━━━━━━━ 14969/14969 2.1Kfiles/s 7.0s
Dataset download success ✅ (14.8s), saved to /content/datasets

val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1464.5±615.1 MB/s, size: 48.0 KB)
val: Scanning /content/datasets/kitti/labels/val... 1496 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1496/1496 1.9Kit/s 0.8s
val: New cache created: /content/datasets/kitti/labels/val.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 94/94 8.4it/s 11.2s
                   all       1496       8128      0.182     0.0487     0.0292     0.0127
Speed: 0.4ms preprocess, 1.9ms inference, 0.0ms 

In [5]:
ARGS = ["--model", "yolov3-tinyu.pt", "--data", "kitti.yaml",
        "--frac-bits", "15", "--out", "q115.json"]
main()

Ultralytics 8.4.211 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLOv3-tiny summary (fused): 34 layers, 12,168,784 parameters, 0 gradients, 19.0 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1045.8±373.8 MB/s, size: 48.7 KB)
val: Scanning /content/datasets/kitti/labels/val.cache... 1496 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1496/1496 348.6Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 94/94 9.0it/s 10.4s
                   all       1496       8128      0.182     0.0487     0.0292     0.0127
Speed: 0.4ms preprocess, 1.7ms inference, 0.0ms loss, 1.3ms postprocess per image
Results saved to /content/runs/detect/val-2
FP32: {'mAP50': 0.02924333681943406, 'mAP50-95': 0.012671339053638953}
YOLOv3-tiny summary (fused): 34 layers, 12,168,784 parameters, 0 gradients, 19.0 GFLOPs
Layers with the largest |w| (need more integer bits):
  model.0.conv: max|w|=27.723, outside Q1 range

In [6]:
ARGS = ["--model", "yolov3-tinyu.pt", "--data", "kitti.yaml",
        "--frac-bits", "7", "--out", "q17.json"]
main()

Ultralytics 8.4.211 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLOv3-tiny summary (fused): 34 layers, 12,168,784 parameters, 0 gradients, 19.0 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2165.0±585.6 MB/s, size: 50.7 KB)
val: Scanning /content/datasets/kitti/labels/val.cache... 1496 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1496/1496 522.9Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 94/94 9.0it/s 10.4s
                   all       1496       8128      0.182     0.0487     0.0292     0.0127
Speed: 0.3ms preprocess, 1.7ms inference, 0.0ms loss, 1.4ms postprocess per image
Results saved to /content/runs/detect/val-4
FP32: {'mAP50': 0.02924333681943406, 'mAP50-95': 0.012671339053638953}
YOLOv3-tiny summary (fused): 34 layers, 12,168,784 parameters, 0 gradients, 19.0 GFLOPs
Layers with the largest |w| (need more integer bits):
  model.0.conv: max|w|=27.723, outside Q1 range

# New Section